# HW4 Decision Trees and Random Forests

In HW3 you clustered posts from four newsgroups without looking at their labels. In this assignment the
newsgroup label is the target, and you will train decision trees to predict it. You will

1. Split the posts into a training set and a test set and compute a majority class baseline.
2. Build feature set A, seven handmade numbers that describe each post.
3. Build feature set B, TFIDF word features, inside a `scikit-learn` pipeline.
4. Compare a decision tree on each feature set.
5. Sweep the maximum tree depth with cross validation and read the training and validation curves.
6. Fit a random forest on the better feature set and look at its feature importances.

Fill in every cell marked **TODO** and answer every question cell. Run the notebook from top to bottom
before submitting so that every table and plot is visible.


**Cell tags.** Every actionable cell is tagged so you know what it expects.
* `[WRITE CODE]` is a code cell for you to fill in.
* `[RUN ONLY]` is provided code. Run it without editing it.
* `[ANSWER]` is a question cell. Type your response where it says *Your answer here.*
* `[REQUIRED]` is a cell you must complete with your information, disclosures, and work history link.
  Leaving these out carries grade penalties, as described under **Submission requirements**.

When a `[WRITE CODE]` or `[RUN ONLY]` cell is followed by an `[ANSWER]` cell, run the code first and then
answer the question about its output.

**[REQUIRED]**

## Student information and disclosures

| | |
|---|---|
| **Name** | *Amit Murkalmath* |
| **Student ID** | *122097179* |
| **Work format** | Individual |

**External source disclosure.** Cite every external source you used and what you got from each one,
including textbooks, websites, Stack Overflow answers, library documentation, YouTube videos, and forum posts.

- Scikit-learn 1.9.1 documentation (`train_test_split`, `TfidfVectorizer`, `Pipeline`, `DecisionTreeClassifier`, `RandomForestClassifier`, `cross_validate`, `accuracy_score`, `f1_score`, `confusion_matrix`): checked parameter signatures and pipeline step syntax.
- Course lecture slides on Supervised Learning, Decision Trees, and Model Evaluation (DATA 602).

**AI usage disclosure.** Describe how you used AI tools such as ChatGPT, Claude, or Copilot, if at all, and
for which parts of the assignment. Any use of AI tools must comply with the course AI policy.

*[YOUR AI DISCLOSURE HERE. Write none only if you truly used none.]*

**[REQUIRED]**

## Work history link

Paste **one** of the following here, following the rules under **Required items** below.

* a **Google Colab** share link with the Anyone with the link setting and **Editor** permission, or
* a **public GitHub repository** URL with an incremental commit history.

https://github.com/LaFlameX07/HW4_Murkalmath_122097179

## Submission requirements

Submit your completed notebook `HW4_[LASTNAME]_[STUDENTID].ipynb`, run from top to bottom with all outputs
visible, together with a PDF export of the same notebook.

### Required items

The following items must be included in the notebook. Leaving any of them out results in a grade deduction.

* **Work history**, an automatic 30 percent deduction from the total assignment grade if omitted
    * **Option A, Google Colab.** Share the live notebook link using the Anyone with the link setting with
      **Editor** permission, and submit it along with a printed PDF of the notebook.
    * **Option B, GitHub.** Host your code in a **public** repository with a visible, incremental commit history.
    * **Authentic timeline.** Your version or commit history must reflect step by step progress. Pasting a
      fully completed project into a blank file at the end results in the automatic loss of this credit.
* **Source citations**, an automatic 30 percent deduction from the total assignment grade if omitted
    * Complete the **External source disclosure** at the top of the notebook.
    * Cite every external source used and what you got from each one, including textbooks, websites,
      Stack Overflow answers, library documentation, YouTube videos, and forum posts.
    * Write none only if you truly used none.
* **AI usage disclosure**, an automatic 30 percent deduction from the total assignment grade if omitted
    * Complete the **AI usage disclosure** at the top of the notebook, describing how AI tools such as
      ChatGPT, Claude, or Copilot were used.
    * Any use of AI tools must comply with the course AI policy.

## Setup

This assignment is pinned to **`scikit-learn` 1.9.1**. The expected results in the rubric come from that
version, and other versions can grow slightly different trees. Run the install cell once. If the version
check in the cell after it fails, choose Restart session from the Runtime menu and run the notebook again
from the top. **Keep the random seed as given** so that your split, trees, and plots are the same every
time you run the notebook.

In [ ]:
# [RUN ONLY]
# Install the pinned scikit-learn version. Run this once per session.
!pip install -q scikit-learn==1.9.1

In [ ]:
# [RUN ONLY]
# Cap threads before importing numpy so results are identical from run to run.
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

print('scikit-learn version', sklearn.__version__)
assert sklearn.__version__ == '1.9.1', (
    'Wrong scikit-learn version. Choose Restart session from the Runtime menu and run again from the top.'
)

# Fix the seed everywhere for reproducible results.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Part 1 Data, split, and baseline

### Dataset

We load the same four newsgroups as HW3, with headers, footers, and quoted replies removed. After that
removal some posts are empty or contain only whitespace. The cell below drops them and prints how many it
dropped from each category. The target for every post is its newsgroup label. The first run downloads the
dataset, so the cell may pause for a few seconds.

In [ ]:
# [RUN ONLY]
from sklearn.datasets import fetch_20newsgroups

categories = ['comp.graphics', 'rec.sport.baseball', 'sci.space', 'rec.autos']
news = fetch_20newsgroups(subset='all', categories=categories,
                          remove=('headers', 'footers', 'quotes'),
                          random_state=RANDOM_STATE)

documents = np.array(news.data, dtype=object)
labels = np.array(news.target)
target_names = news.target_names

# Drop posts that are empty or contain only whitespace after the removal step.
is_empty = np.array([doc.strip() == '' for doc in documents])
counts = pd.DataFrame({'loaded': np.bincount(labels, minlength=4),
                       'empty': np.bincount(labels[is_empty], minlength=4)},
                      index=target_names)
counts['kept'] = counts['loaded'] - counts['empty']
print(counts.to_string())

documents = documents[~is_empty]
labels = labels[~is_empty]
print(f'\n{len(documents)} posts kept across {len(target_names)} categories.')

### TODO 1 Stratified split and majority class baseline

Split `documents` and `labels` into a training set and a test set.
* Put 25 percent of the posts in the test set.
* Stratify on the labels so that each category has the same share in both sets.
* Use `random_state=RANDOM_STATE`.
* Name the results `docs_train`, `docs_test`, `y_train`, and `y_test`.

Then compute the majority class baseline. This baseline always predicts the category that is most common in
the **training** set. Print which category that is and the accuracy of the baseline on the test set, and
store that accuracy in `baseline_acc`.

In [ ]:
# [WRITE CODE]
from sklearn.model_selection import train_test_split
#####################################################################
# TODO                                                              #
# 1. Split documents and labels with test_size=0.25, stratified on  #
#    the labels, with random_state=RANDOM_STATE. Name the results   #
#    docs_train, docs_test, y_train, y_test.                        #
# 2. Find the most common category in y_train.                      #
# 3. Compute the test accuracy of always predicting that category,  #
#    store it in baseline_acc, and print the category and accuracy. #
pass
#####################################################################
#                       END OF YOUR CODE                            #
#####################################################################

In [ ]:
# [RUN ONLY]
# Checkpoint for TODO 1.
assert len(docs_train) + len(docs_test) == len(documents), 'Every post should land in exactly one set.'
assert abs(len(docs_test) / len(documents) - 0.25) < 0.01, 'The test set should hold 25 percent of the posts.'
assert 0 < baseline_acc < 1, 'baseline_acc should be an accuracy between 0 and 1.'
print(pd.DataFrame({'train': np.bincount(y_train), 'test': np.bincount(y_test)},
                   index=target_names).to_string())
print('TODO 1 checkpoint passed.')

# Part 2 Feature set A, handmade features


### TODO 2 Seven handmade features

Write a function that turns the text of one post into the seven numbers below, in this order. A word is any
piece of text between whitespace, so `Hello,` and `world!` count as two words.

1. `n_chars` is the number of characters in the post, including spaces and line breaks.
2. `n_words` is the number of words.
3. `n_lines` is the number of lines you get when you break the text at each line break.
4. `n_digits` is the number of characters that are digits 0 through 9.
5. `upper_ratio` is the number of uppercase letters divided by the number of letters. Use 0 for a post
   with no letters.
6. `n_question_marks` is the number of question mark characters.
7. `avg_word_len` is the average number of characters per word.

**Rules.** Each feature must be computed from the text of that one post alone. No feature may use the
label, the category name, or any other post.

**No scaling.** Do not scale or standardize these features, even though they are on very different scales.
A tree splits on one feature at a time by comparing it with a threshold, so the units of a feature do not
change which splits are possible. This is a contrast with k means and KNN from L08, which compare posts by
distance, where a feature in the thousands such as `n_chars` would swamp a ratio between 0 and 1.

After the function passes the checkpoint, build `A_train` and `A_test`, which are DataFrames with one row per
post and the seven columns in `FEATURE_NAMES` order. Finally, show a table of the median of each feature in
each category, computed on the training set only and labeled with category names.

In [ ]:
# [WRITE CODE]
FEATURE_NAMES = ['n_chars', 'n_words', 'n_lines', 'n_digits',
                 'upper_ratio', 'n_question_marks', 'avg_word_len']

def handmade_features(text):
    """Return a list of the seven feature values for one post, in FEATURE_NAMES order."""
    #####################################################################
    # TODO                                                              #
    # Compute the seven features defined above from text alone and      #
    # return them as a list of seven numbers in FEATURE_NAMES order.    #
    pass
    #####################################################################
    #                       END OF YOUR CODE                            #
    #####################################################################

In [ ]:
# [RUN ONLY]
# Checkpoint for handmade_features on two small posts with known answers.
assert np.allclose(handmade_features('Is it 42?\nYES'), [13, 4, 2, 2, 4 / 7, 1, 2.5]), \
    'Your features do not match the expected values for the first example.'
assert np.allclose(handmade_features('123 456'), [7, 2, 1, 6, 0, 0, 3]), \
    'Your features do not match the expected values for the second example.'
print('handmade_features checkpoint passed.')

In [ ]:
# [WRITE CODE]
#####################################################################
# TODO                                                              #
# 1. Build A_train and A_test, DataFrames with one row per post and #
#    the columns FEATURE_NAMES.                                     #
# 2. Show the median of each feature in each category, computed on  #
#    the training set only, labeled with category names.            #
pass
#####################################################################
#                       END OF YOUR CODE                            #
#####################################################################

In [ ]:
# [RUN ONLY]
# Checkpoint for TODO 2.
assert list(A_train.columns) == FEATURE_NAMES and list(A_test.columns) == FEATURE_NAMES
assert A_train.shape == (len(docs_train), 7) and A_test.shape == (len(docs_test), 7)
assert np.isfinite(A_train.to_numpy(dtype=float)).all() and np.isfinite(A_test.to_numpy(dtype=float)).all()
print('TODO 2 checkpoint passed.')

# Part 3 Feature set B, TFIDF words


### TODO 3 TFIDF and a tree in one pipeline

Build a `scikit-learn` `Pipeline` named `tree_B` with two steps.
1. A step named `'tfidf'` holding `TfidfVectorizer(stop_words='english', max_features=1000)`.
2. A step named `'tree'` holding `DecisionTreeClassifier(random_state=RANDOM_STATE)` with every other
   setting left at its default.

Fit `tree_B` on the raw training posts `docs_train` with labels `y_train`. Then print the shape of the TFIDF
matrix that the fitted vectorizer produces for the training posts.

In [ ]:
# [WRITE CODE]
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
#####################################################################
# TODO                                                              #
# 1. Build tree_B, a Pipeline with a step named 'tfidf' holding     #
#    TfidfVectorizer(stop_words='english', max_features=1000) and a #
#    step named 'tree' holding                                      #
#    DecisionTreeClassifier(random_state=RANDOM_STATE).             #
# 2. Fit tree_B on docs_train and y_train.                          #
# 3. Print the shape of the TFIDF matrix for the training posts.    #
pass
#####################################################################
#                       END OF YOUR CODE                            #
#####################################################################

# Part 4 Comparing the two trees

### TODO 4 Tree on A, tree on B, and the baseline

Fit `tree_A`, a `DecisionTreeClassifier(random_state=RANDOM_STATE)` with every other setting left at its
default, on `A_train`. Predict the test set with both trees and store the predictions in `pred_A` and
`pred_B`.

Macro F1 is the average of the F1 scores of the four categories, with every category weighted equally.
Show two tables.
1. A comparison table with three rows, the majority baseline, the tree on A, and the tree on B, and two
   columns, test accuracy and test macro F1.
2. The confusion matrices of the two trees side by side in one table. Rows are true categories and columns
   are predicted categories, both labeled with category names.

In [ ]:
# [WRITE CODE]
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
#####################################################################
# TODO                                                              #
# 1. Fit tree_A on A_train and y_train.                             #
# 2. Store test predictions in pred_A and pred_B.                   #
# 3. Show the comparison table of accuracy and macro F1 for the     #
#    majority baseline, the tree on A, and the tree on B.           #
# 4. Show both confusion matrices side by side in one table, with   #
#    true categories as rows and predicted categories as columns.   #
pass
#####################################################################
#                       END OF YOUR CODE                            #
#####################################################################

### Misclassified posts

The next cell prints six test posts that the tree on B gets wrong, chosen at random with the fixed seed,
along with the true and predicted category of each. You will use them in Question 1.

In [ ]:
# [RUN ONLY]
wrong = np.flatnonzero(pred_B != y_test)
rng = np.random.default_rng(RANDOM_STATE)
for i in rng.choice(wrong, size=6, replace=False):
    print(f'Test post {i}, true category {target_names[y_test[i]]}, '
          f'predicted {target_names[pred_B[i]]}')
    print(docs_test[i][:400].strip())
    print('-' * 70)

**[ANSWER]**

### Question 1 Comparing the feature sets, 2 points

Use the comparison table, the confusion matrices, your table of per category medians from TODO 2, and the
misclassified posts above.

* State which feature set gives the better tree and by how much, using accuracy and macro F1, and say how each
  tree compares with the baseline.
* Explain why the tree on A does so much worse. Support your explanation with your table of per category
  medians.
* Describe where the errors of the better tree go. Say whether they are concentrated in a few pairs of
  categories or spread across many, and give counts from the confusion matrix.
* Read the misclassified posts. Describe what several of them have in common and explain why a single tree
  on word features gets them wrong.

*Your answer here.*

# Part 5 Depth sweep


### TODO 5 Training and validation accuracy across depths

For each value of `max_depth` in `DEPTHS` and for each feature set, run 5 fold cross validation **on the
training set only**. Record the mean training accuracy and the mean validation accuracy over the five folds.
* For feature set A, cross validate `DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE)` on
  `A_train`.
* For feature set B, cross validate the whole TFIDF and tree pipeline, with the same tree settings, on
  `docs_train`.
* Use `cv=5` so that everyone gets the same folds. A depth of `None` means no depth limit.

Store the results in two DataFrames, `sweep_A` and `sweep_B`, each with the columns `depth`, `train_acc`,
and `val_acc`, and print both. Make one figure with two panels, one per feature set, each showing training
and validation accuracy against depth. Put the depths on the x axis in the order of `DEPTHS`, with the
unlimited setting labeled None, and label both axes.

Finally, store the depth with the highest mean validation accuracy for each feature set in `best_depth_A`
and `best_depth_B`. Each must be one of the values in `DEPTHS`, so either an integer or `None`.

This cell takes about a minute to run, mostly for feature set B.

In [ ]:
# [WRITE CODE]
from sklearn.model_selection import cross_validate

DEPTHS = [1, 2, 3, 4, 5, 6, 8, 10, 15, 20, 30, 50, 100, None]
#####################################################################
# TODO                                                              #
# 1. For every depth in DEPTHS, run 5 fold cross validation with    #
#    cv=5 on the training set for feature set A and feature set B,  #
#    recording mean training and mean validation accuracy.          #
# 2. Store the results in sweep_A and sweep_B with the columns      #
#    depth, train_acc, val_acc, and print both.                     #
# 3. Plot training and validation accuracy against depth in one     #
#    figure with one panel per feature set.                         #
# 4. Store best_depth_A and best_depth_B.                           #
pass
#####################################################################
#                       END OF YOUR CODE                            #
#####################################################################

In [ ]:
# [RUN ONLY]
# Checkpoint for TODO 5.
for name, table in [('sweep_A', sweep_A), ('sweep_B', sweep_B)]:
    assert list(table.columns) == ['depth', 'train_acc', 'val_acc'], f'{name} has the wrong columns.'
    assert len(table) == len(DEPTHS), f'{name} should have one row per depth.'
assert best_depth_A in DEPTHS and best_depth_B in DEPTHS, 'Each best depth must be a value from DEPTHS.'
print('TODO 5 checkpoint passed.')

**[ANSWER]**

### Question 2 Reading the depth sweep

Use the two sweep tables and the figure.

* For each feature set, give the depth with the highest mean validation accuracy and that accuracy, and
  describe what validation accuracy does at greater depths.
* Explain why the two validation curves behave differently. A complete answer names a property of each
  feature set that causes its behavior, not only a description of the curves.
* With no depth limit, the tree on B still does not reach a training accuracy of 1.0. Explain how a tree with
  no depth limit can make errors on its own training data here.

*Your answer here.*

# Part 6 Random forest


### TODO 6 A forest against the best single tree

Pick the feature set with the higher best mean validation accuracy in your sweep. On that feature set
* build `best_tree`, a single `DecisionTreeClassifier` with `max_depth` set to the best depth you found for
  that feature set and `random_state=RANDOM_STATE`, inside the same kind of pipeline as before if you
  picked B, and fit it on the training set,
* build `forest`, a `RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)`, inside the same
  kind of pipeline if you picked B, and fit it on the training set,
* show a table with the test accuracy and test macro F1 of `best_tree` and `forest`,
* show the 10 features with the largest importances in `forest`, with the name of each feature, as a table
  or a bar chart.

**About these importances.** The `feature_importances_` of a forest add up how much each feature reduced
impurity over all splits in all trees. Impurity based importances can be biased, for example toward
features with many distinct values, so treat them as a rough guide to what the forest relies on and not as
proof of what matters.

In [ ]:
# [WRITE CODE]
from sklearn.ensemble import RandomForestClassifier
#####################################################################
# TODO                                                              #
# 1. Pick the feature set with the higher best validation accuracy. #
# 2. Build and fit best_tree and forest on that feature set.        #
# 3. Show test accuracy and macro F1 for both in one table.         #
# 4. Show the 10 largest feature importances of forest by name.     #
pass
#####################################################################
#                       END OF YOUR CODE                            #
#####################################################################

# Part 7 Leakage

Leakage happens when information that would not be available at prediction time, such as the test posts or
the labels, influences how a model is trained or chosen.

**[ANSWER]**

### Question 3 One leakage risk

* Name one specific leakage risk in this assignment and point to the step in this notebook where it could
  happen. A general definition of leakage does not count.
* Explain how it would change the reported test numbers if it happened.
* Explain how this notebook prevents it, or what you would change to prevent it if it does not.

*Your answer here.*

### Optional, not graded. Features of your own

If you want to explore, design one or two extra features from the text of a post and check whether they help
the tree on A. Keep them in a separate DataFrame so that feature set A stays exactly the seven listed in
TODO 2. For each feature, write one sentence saying whether it measures what the post is about or how the
post was written and formatted.

In [ ]:
# [WRITE CODE] Optional, not graded.

---
### Submission

Run the notebook from top to bottom with all outputs visible, answer every question cell, and submit this
notebook together with its PDF export as described under **Submission requirements**.